In [0]:
# =====================================================
# PARAMÉTRAGE - Widgets pour exécution via Databricks Job
# =====================================================

dbutils.widgets.text("catalog_name", "banking_lakehouse", "Catalog Unity Catalog")
dbutils.widgets.text("environment", "dev", "Environnement (dev/staging/prod)")

CATALOG = dbutils.widgets.get("catalog_name")
ENVIRONMENT = dbutils.widgets.get("environment")

print(f"✅ Paramètres reçus : catalog={CATALOG}, environment={ENVIRONMENT}")

In [0]:
# =====================================================
# Notebook : 03_transform_silver_clients
# Objectif : Nettoyer, dédupliquer et fiabiliser les données
#            clients (Bronze -> Silver) avec Data Quality
#            et MERGE INTO (upsert incrémental)
# Domaine  : Banking Lakehouse
# =====================================================

from pyspark.sql.functions import (
    col, current_timestamp, row_number, when, lit, upper, trim
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- Configuration ---
TABLE_CLIENTS_BRONZE = f"{CATALOG}.bronze.clients_raw"
TABLE_CLIENTS_SILVER = f"{CATALOG}.silver.clients"

# Référentiel des valeurs valides (bonne pratique : centraliser les règles métier)
VALID_GEOGRAPHIES = ["France", "Germany", "Spain"]
MIN_CREDIT_SCORE, MAX_CREDIT_SCORE = 300, 900
MIN_AGE, MAX_AGE = 18, 100

print("✅ Configuration Silver chargée")
print(f"Source Bronze : {TABLE_CLIENTS_BRONZE}")
print(f"Cible Silver  : {TABLE_CLIENTS_SILVER}")

In [0]:
# =====================================================
# Lecture Bronze + Déduplication sur CustomerId
# =====================================================

df_bronze = spark.table(TABLE_CLIENTS_BRONZE)

print(f"📊 Nombre de lignes en Bronze (avant dédup) : {df_bronze.count()}")

# --- Déduplication : on garde la ligne la plus récente par CustomerId ---
window_dedup = Window.partitionBy("CustomerId").orderBy(col("_ingestion_timestamp").desc())

df_deduped = (
    df_bronze
    .withColumn("_row_num", row_number().over(window_dedup))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

print(f"📊 Nombre de lignes après déduplication : {df_deduped.count()}")

nb_doublons_supprimes = df_bronze.count() - df_deduped.count()
print(f"🔎 Doublons supprimés : {nb_doublons_supprimes}")

display(df_deduped.limit(5))

In [0]:
# =====================================================
# Application des règles de Data Quality (DQ)
# Approche : flags qualité (non-destructif)
# =====================================================

df_quality = (
    df_deduped
    # --- Nettoyage léger ---
    .withColumn("Geography", trim(col("Geography")))
    .withColumn("Gender", trim(upper(col("Gender"))))
    
    # --- Flags qualité individuels ---
    .withColumn(
        "dq_valid_customerid",
        when(col("CustomerId").isNotNull(), lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "dq_valid_creditscore",
        when(
            (col("CreditScore") >= MIN_CREDIT_SCORE) & (col("CreditScore") <= MAX_CREDIT_SCORE),
            lit(True)
        ).otherwise(lit(False))
    )
    .withColumn(
        "dq_valid_age",
        when(
            (col("Age") >= MIN_AGE) & (col("Age") <= MAX_AGE),
            lit(True)
        ).otherwise(lit(False))
    )
    .withColumn(
        "dq_valid_balance",
        when(col("Balance") >= 0, lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "dq_valid_geography",
        when(col("Geography").isin(VALID_GEOGRAPHIES), lit(True)).otherwise(lit(False))
    )
    
    # --- Flag qualité global (synthèse) ---
    .withColumn(
        "dq_is_valid",
        col("dq_valid_customerid") &
        col("dq_valid_creditscore") &
        col("dq_valid_age") &
        col("dq_valid_balance") &
        col("dq_valid_geography")
    )
    
    # --- Métadonnée de traçabilité Silver ---
    .withColumn("_silver_processed_at", current_timestamp())
)

# --- Rapport de qualité ---
print("📋 Rapport de Data Quality :")
df_quality.select(
    "dq_valid_customerid", "dq_valid_creditscore", "dq_valid_age",
    "dq_valid_balance", "dq_valid_geography", "dq_is_valid"
).groupBy().agg(
    (col("dq_valid_customerid") == False).cast("int").alias("tmp")  # placeholder
).show() if False else None

nb_total = df_quality.count()
nb_valid = df_quality.filter(col("dq_is_valid") == True).count()
nb_invalid = nb_total - nb_valid

print(f"📊 Total lignes         : {nb_total}")
print(f"✅ Lignes valides        : {nb_valid} ({round(100*nb_valid/nb_total, 2)}%)")
print(f"⚠️  Lignes avec anomalie : {nb_invalid} ({round(100*nb_invalid/nb_total, 2)}%)")

# --- Détail des anomalies s'il y en a ---
if nb_invalid > 0:
    print("\n🔎 Détail des lignes invalides :")
    df_quality.filter(col("dq_is_valid") == False).select(
        "CustomerId", "CreditScore", "Age", "Balance", "Geography",
        "dq_valid_creditscore", "dq_valid_age", "dq_valid_balance", "dq_valid_geography"
    ).show(20, truncate=False)

In [0]:
# =====================================================
# MERGE INTO - Upsert vers Silver (CDC manuel)
# Logique : 
#   - Si CustomerId existe déjà en Silver -> UPDATE (si donnée plus récente)
#   - Si CustomerId n'existe pas encore -> INSERT
# =====================================================

# On ne garde que les lignes valides pour l'instant
# (stratégie : lignes invalides en quarantaine, à gérer plus tard si besoin)
df_to_merge = df_quality.filter(col("dq_is_valid") == True)

# --- Vérifier si la table Silver existe déjà ---
table_exists = spark.catalog.tableExists(TABLE_CLIENTS_SILVER)

if not table_exists:
    print(f"🆕 Table {TABLE_CLIENTS_SILVER} n'existe pas encore -> création initiale")
    df_to_merge.write.format("delta").saveAsTable(TABLE_CLIENTS_SILVER)
    print(f"✅ Table Silver créée avec {df_to_merge.count()} lignes")
else:
    print(f"🔄 Table {TABLE_CLIENTS_SILVER} existe -> MERGE INTO (upsert)")
    
    delta_table_silver = DeltaTable.forName(spark, TABLE_CLIENTS_SILVER)
    
    (
        delta_table_silver.alias("target")
        .merge(
            df_to_merge.alias("source"),
            "target.CustomerId = source.CustomerId"
        )
        .whenMatchedUpdate(
            condition="source._ingestion_timestamp > target._ingestion_timestamp",
            set={
                "CreditScore": "source.CreditScore",
                "Geography": "source.Geography",
                "Gender": "source.Gender",
                "Age": "source.Age",
                "Tenure": "source.Tenure",
                "Balance": "source.Balance",
                "NumOfProducts": "source.NumOfProducts",
                "HasCrCard": "source.HasCrCard",
                "IsActiveMember": "source.IsActiveMember",
                "EstimatedSalary": "source.EstimatedSalary",
                "Exited": "source.Exited",
                "Country_Risk_Level": "source.Country_Risk_Level",
                "_ingestion_timestamp": "source._ingestion_timestamp",
                "_silver_processed_at": "source._silver_processed_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    
    print(f"✅ MERGE INTO exécuté avec succès")

# --- Vérification finale ---
df_silver_check = spark.table(TABLE_CLIENTS_SILVER)
print(f"\n📊 Nombre total de lignes en Silver : {df_silver_check.count()}")

In [0]:
# =====================================================
# TEST CDC : simuler la mise à jour d'un client existant
# Scénario : le client 15634602 (Hargrave) voit son
# CreditScore et Balance évoluer (ex: recalcul scoring)
# Approche : DataFrame transformation directe (pas de Row Python)
# =====================================================

print("📋 AVANT modification :")
df_silver_check.filter(col("CustomerId") == 15634602).select(
    "CustomerId", "Surname", "CreditScore", "Balance", "_ingestion_timestamp"
).show()

# On part de la ligne existante en Silver, et on la modifie via withColumn
df_update_test = (
    df_silver_check
    .filter(col("CustomerId") == 15634602)
    .withColumn("CreditScore", lit(750))              # ancien : 619
    .withColumn("Balance", lit(45000.00))              # ancien : 0.0
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_silver_processed_at", current_timestamp())
    # On retire les colonnes de flags DQ (seront recalculées par le prochain passage qualité)
    .drop("dq_valid_customerid", "dq_valid_creditscore", "dq_valid_age",
          "dq_valid_balance", "dq_valid_geography", "dq_is_valid")
)

print("\n📋 Nouvelle version à merger :")
df_update_test.select("CustomerId", "Surname", "CreditScore", "Balance", "_ingestion_timestamp").show()

In [0]:
# =====================================================
# MERGE INTO réel - Application de la mise à jour CDC
# (Test ciblé : uniquement branche UPDATE, le client existe déjà)
# =====================================================

delta_table_silver = DeltaTable.forName(spark, TABLE_CLIENTS_SILVER)

nb_lignes_avant = spark.table(TABLE_CLIENTS_SILVER).count()

(
    delta_table_silver.alias("target")
    .merge(
        df_update_test.alias("source"),
        "target.CustomerId = source.CustomerId"
    )
    .whenMatchedUpdate(
        condition="source._ingestion_timestamp > target._ingestion_timestamp",
        set={
            "CreditScore": "source.CreditScore",
            "Balance": "source.Balance",
            "_ingestion_timestamp": "source._ingestion_timestamp",
            "_silver_processed_at": "source._silver_processed_at"
        }
    )
    .execute()   # 🆕 on retire whenNotMatchedInsertAll() pour ce test ciblé
)

nb_lignes_apres = spark.table(TABLE_CLIENTS_SILVER).count()

print(f"📊 Nombre de lignes AVANT le merge : {nb_lignes_avant}")
print(f"📊 Nombre de lignes APRÈS le merge : {nb_lignes_apres}")
print(f"✅ Différence : {nb_lignes_apres - nb_lignes_avant} (doit être 0 -> UPDATE, pas INSERT)")

print("\n📋 Vérification du client 15634602 APRÈS le merge :")
spark.table(TABLE_CLIENTS_SILVER).filter(col("CustomerId") == 15634602).select(
    "CustomerId", "Surname", "CreditScore", "Balance", "_ingestion_timestamp"
).show()